In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px

model_dir = r"C:\workspace\sc_asim_2023_08_03_26"
output_dir = r"C:\workspace\sc_asim_2023_08_03_26_output_NEW_NEST"
data_dir = r"C:\workspace\sc_asim_2023_08_03_26_data"

df = pd.read_parquet(f"{output_dir}\\final_trips.parquet")
df_tour = pd.read_parquet(f"{output_dir}\\final_tours.parquet")
land_use = pd.read_csv(f"{data_dir}\\land_use.csv")
pnr_capacity = pd.read_csv(f"{model_dir}\\inputs\\scenario\\networks\\p_r_nodes.csv")

In [2]:
df["trip_mode"].value_counts()

trip_mode
DRIVEALONEFREE    6073213
SHARED2FREE       2427694
SHARED3FREE       1638868
WALK              1560403
BIKE               234629
SCH_BUS            227333
TNC                130486
WALK_LOC           108322
WALK_LR             82064
DRIVE_TRN           41308
WALK_COM             9279
WALK_FRY             6655
                        0
Name: count, dtype: int64

In [28]:
import os
import pandas as pd
import numpy as np
from sqlalchemy import create_engine,text
from scipy import stats
import plotly.express as px
import toml
import geopandas as gpd
import plotly.express as px
import plotly.graph_objects as go
import sys

sys.path.append("../../notebook_styling")
import psrc_theme

input_config = toml.load(os.path.join(os.getcwd(), '..\\..\\..\\..\\configuration', 'input_configuration.toml'))
valid_config = toml.load(os.path.join(os.getcwd(), '..\\..\\..\\..\\configuration', 'validation_configuration.toml'))
summary_config = toml.load(os.path.join(os.getcwd(), '..\\..\\..\\..\\configuration', 'summary_configuration.toml'))

# create connection
conn = create_engine('sqlite:///../../../../inputs/db/'+input_config['db_name'])
# summary data location
csv_path = os.path.join(valid_config['model_dir'], 'outputs/validation')

# table format
pd.options.display.float_format = '{:0,.0f}'.format
format_percent = "{:,.2%}".format

In [29]:
df_tour["tour_mode"].value_counts()

tour_mode
DRIVEALONEFREE    1879998
SHARED2FREE       1324050
SHARED3FREE        761759
WALK               556402
SCH_BUS            139803
BIKE                77133
TNC                 54690
WALK_LR             30312
WALK_LOC            25240
DRIVE_TRN           23679
WALK_COM             6524
WALK_FRY             4102
                        0
Name: count, dtype: int64

In [3]:
df_tour_drive_trn = df_tour[df_tour["tour_mode"] == "DRIVE_TRN"]

df_tour_taz = df_tour_drive_trn.merge(land_use[["MAZ", "TAZ"]], left_on="pnr_zone_id", right_on="MAZ", how="left")

taz_counts = df_tour_taz["TAZ"].value_counts().reset_index()
taz_counts.columns = ["TAZ", "count"]

pnr_boardings = df_tour_taz[df_tour_taz["TAZ"].isin(range(3750, 4001))]

pnr_boardings_count = pnr_boardings["TAZ"].value_counts().reset_index()
pnr_boardings_count.columns = ["TAZ", "count"]

In [10]:
stations = [
    {"id": 3751, "name": "Tukwila Station (CR)"},
    {"id": 3752, "name": "Tacoma Dome Station"},
    {"id": 3753, "name": "South Tacoma (CR)"},
    {"id": 3754, "name": "Everett Station (CR)"},
    {"id": 3755, "name": "Sumner Station (CR)"},
    {"id": 3756, "name": "Mukilteo Station"},
    {"id": 3757, "name": "Kent Station (CR)"},
    {"id": 3758, "name": "Auburn Station (CR)"},
    {"id": 3759, "name": "Lakewood (CR)"},
    {"id": 3760, "name": "Gateway P&R"},
    {"id": 3761, "name": "Puyallup Station (SB)"},
    {"id": 3762, "name": "Twin Lakes P&R"},
    {"id": 3763, "name": "Crossroads Neighborhood Church"},
    {"id": 3764, "name": "Overlake Transit Center"},
    {"id": 3765, "name": "Redondo Heights P&R"},
    {"id": 3766, "name": "North Gig Harbor/Kimball Drive"},
    {"id": 3767, "name": "Duvall P&R"},
    {"id": 3768, "name": "Roy 'Y' P&R"},
    {"id": 3769, "name": "South Hill (Elim Evangelical)"},
    {"id": 3770, "name": "Parkland Transit Center"},
    {"id": 3771, "name": "SR-512/I-5 (Lakewood)"},
    {"id": 3772, "name": "Center Street P&R"},
    {"id": 3773, "name": "Tacoma Mall Transit Center"},
    {"id": 3774, "name": "South Tacoma West"},
    {"id": 3775, "name": "South Tacoma East - 2"},
    {"id": 3776, "name": "Narrows P&R"},
    {"id": 3777, "name": "North Purdy/Purdy Crescent"},
    {"id": 3778, "name": "Point Defiance Ferry"},
    {"id": 3779, "name": "South Federal P&R"},
    {"id": 3780, "name": "Sultan P&R"},
    {"id": 3781, "name": "Federal Way P&R (320th)"},
    {"id": 3782, "name": "Auburn P&R"},
    {"id": 3783, "name": "Star Lake P&R"},
    {"id": 3784, "name": "Holy Spirit Lutheran Church"},
    {"id": 3785, "name": "Kent/Des Moines P&R"},
    {"id": 3786, "name": "Kent/James St. P&R"},
    {"id": 3787, "name": "Burien Transit Center"},
    {"id": 3788, "name": "Tukwila"},
    {"id": 3789, "name": "Maple Valley P&R"},
    {"id": 3790, "name": "South Renton P&R"},
    {"id": 3791, "name": "Renton Highlands P&R (St. Matts)"},
    {"id": 3792, "name": "Issaquah Transit Center"},
    {"id": 3793, "name": "Mercer Island P&R"},
    {"id": 3794, "name": "Mercer Island Presbyterian"},
    {"id": 3795, "name": "Newport Hills P&R"},
    {"id": 3796, "name": "Newport Hills Community Church"},
    {"id": 3797, "name": "Newport Covenant Church"},
    {"id": 3798, "name": "South Bellevue P&R"},
    {"id": 3799, "name": "Wilburton"},
    {"id": 3800, "name": "Eastgate P&R"},
    {"id": 3801, "name": "North Bend P&R"},
    {"id": 3802, "name": "Evergreen Point P&R"},
    {"id": 3803, "name": "Grace Lutheran Church"},
    {"id": 3804, "name": "South Kirkland P&R"},
    {"id": 3805, "name": "Overlake P&R"},
    {"id": 3806, "name": "Houghton P&R"},
    {"id": 3807, "name": "Bethel Lutheran"},
    {"id": 3808, "name": "Redmond P&R"},
    {"id": 3809, "name": "Saint Thomas Episcopal"},
    {"id": 3810, "name": "Valley Center"},
    {"id": 3811, "name": "Bear Creek P&R"},
    {"id": 3812, "name": "Kingsgate P&R"},
    {"id": 3813, "name": "Brickyard P&R"},
    {"id": 3814, "name": "Northshore P&R"},
    {"id": 3815, "name": "Kenmore P&R"},
    {"id": 3816, "name": "Bethany Bible Church"},
    {"id": 3817, "name": "Bothell P&R"},
    {"id": 3818, "name": "Woodinville P&R"},
    {"id": 3819, "name": "Olson & Meyers"},
    {"id": 3820, "name": "Spokane Street P&R"},
    {"id": 3821, "name": "Greenlake P&R"},
    {"id": 3822, "name": "Northgate TC"},
    {"id": 3823, "name": "North Jackson Park"},
    {"id": 3824, "name": "Shoreline"},
    {"id": 3825, "name": "112th St. & I-5"},
    {"id": 3826, "name": "Ober Park"},
    {"id": 3827, "name": "Vashon Heights"},
    {"id": 3828, "name": "Mountlake Terrace P&R"},
    {"id": 3829, "name": "Bethesda Lutheran"},
    {"id": 3830, "name": "Edmonds Station"},
    {"id": 3831, "name": "Edmonds P&R"},
    {"id": 3832, "name": "Lynnwood P&R"},
    {"id": 3833, "name": "Swamp Creek P&R"},
    {"id": 3834, "name": "Mariner P&R"},
    {"id": 3835, "name": "Snohomish"},
    {"id": 3836, "name": "Marysville - Ash Ave"},
    {"id": 3837, "name": "116th & I-5 - Marysville"},
    {"id": 3838, "name": "SR-531 - Marysville"},
    {"id": 3839, "name": "Arlington P&R"},
    {"id": 3840, "name": "Stanwood"},
    {"id": 3841, "name": "Monroe"},
    {"id": 3842, "name": "Federal Way"},
    {"id": 3843, "name": "Port Orchard Armory"},
    {"id": 3844, "name": "Southworth Ferry P&R"},
    {"id": 3845, "name": "McWilliams P&R"},
    {"id": 3846, "name": "Bethany Lutheran Church"},
    {"id": 3847, "name": "Gateway Fellowship"},
    {"id": 3848, "name": "Agate Pass P&R/Clearwater"},
    {"id": 3849, "name": "Kingston Ferry P&R"},
    {"id": 3850, "name": "Suquamish United Church of Christ"},
    {"id": 3851, "name": "Bainbridge Island Ferry"},
    {"id": 3852, "name": "Puyallup Station (NB)"},
    {"id": 3853, "name": "72nd Street Transit Center"},
    {"id": 3855, "name": "DuPont"},
    {"id": 3856, "name": "Aurora Village TC"},
    {"id": 3857, "name": "Aurora Nazarene"},
    {"id": 3858, "name": "Saint Margaret's Episcopal"},
    {"id": 3861, "name": "Lake Meridian P&R"},
    {"id": 3862, "name": "Smokey Pt. Church"},
    {"id": 3863, "name": "Eastmont P&R"},
    {"id": 3864, "name": "McCollum Park P&R"},
    {"id": 3865, "name": "Canyon Park P&R"},
    {"id": 3866, "name": "Korean Presby. P&R"},
    {"id": 3867, "name": "Georges Kountry Korner"},
    {"id": 3868, "name": "Poulsbo Church of the Nazarene"},
    {"id": 3870, "name": "Harper Evangelical Free Church"},
    {"id": 3871, "name": "Mullenix Road"},
    {"id": 3873, "name": "TCC P&R"},
    {"id": 3874, "name": "Ash Way P&R"},
    {"id": 3875, "name": "Bonney Lake P&R"},
    {"id": 3876, "name": "Tukwila Int. Station"},
    {"id": 3877, "name": "South Hill P&R"},
    {"id": 3878, "name": "South Tacoma East - 1"},
    {"id": 3879, "name": "Issaquah Highlands"},
    {"id": 3880, "name": "Lake Stevens TC"},
    {"id": 3881, "name": "Liberty Bay Presbyterian"},
    {"id": 3882, "name": "NK Baptist"},
    {"id": 3883, "name": "Preston P&R"},
    {"id": 3884, "name": "Bayside Community Church"},
    {"id": 3885, "name": "Ollala Valley Fire Station"},
    {"id": 3886, "name": "Bremerton Ferry"},
    {"id": 3888, "name": "Puyallup Fair's Red Lot"},
    {"id": 3889, "name": "Marysville - Cedar & Grove"},
    {"id": 3890, "name": "Stanwood II"},
    {"id": 3891, "name": "Marysville I P&R"},
    {"id": 3892, "name": "Martha Lake Covenant Church"},
    {"id": 3893, "name": "South Sammammish P&R"},
    {"id": 3894, "name": "Renton City Municipal Garage"},
    {"id": 3895, "name": "Tibbetts Lot"},
    {"id": 3896, "name": "Renton Transit Center"},
    {"id": 3897, "name": "Calvary Christian Assembly"},
    {"id": 3898, "name": "Maple Valley Town Square"},
    {"id": 3899, "name": "All Saints Lutheran Church"},
    {"id": 3900, "name": "City View Church"},
    {"id": 3901, "name": "Northwest University 6710 Bldg."},
    {"id": 3902, "name": "Sammamish Lutheran P&R"},
    {"id": 3903, "name": "Redmond Ridge P&R"},
    {"id": 3904, "name": "Kennydale United Methodist Church"},
    {"id": 3905, "name": "Nativity Lutheran Church"},
    {"id": 3906, "name": "South Jackson P&R"},
    {"id": 3907, "name": "South SeaTac (LR)"},
    {"id": 3908, "name": "Wheaton Way"},
    {"id": 3909, "name": "Bremerton (SR-303/Riddell Road)"},
    {"id": 3910, "name": "Silverdale"},
    {"id": 3911, "name": "Tukwila Station (CR)"},
    {"id": 3913, "name": "Sumner Station Garage"},
    {"id": 3914, "name": "Tacoma Dome"},
    {"id": 3915, "name": "Bothell (SR-527/185th Street)"},
    {"id": 3916, "name": "SR-3/SR-303"},
    {"id": 3917, "name": "SR-16/SR-160"},
    {"id": 3918, "name": "I-5 & 175th"}
]

In [11]:
df = pnr_capacity.merge(pnr_boardings_count, left_on='ZoneID', right_on='TAZ')
df.rename(columns={'ZoneID': 'taz', 'count': 'Model Boardings'}, inplace=True)

df['Boardings/Capacity'] = df['Model Boardings']/df['Capacity']

df_station_names = pd.DataFrame(stations)
df_with_stations = df.merge(df_station_names, how='left', left_on='TAZ', right_on='id')

In [12]:
# df_with_stations.to_csv("C:\\Users\\modeller\\Desktop\\df_with_stations.csv", index=False)

In [ ]:
df_with_stations['color'] = np.where(df_with_stations['Boardings/Capacity'] > 1, 'Boardings/Capacity > 1', 'Boardings/Capacity <= 1')

max_val = max(
    df_with_stations['Capacity'].replace([np.inf, -np.inf], np.nan).dropna().max(),
    df_with_stations['Model Boardings'].replace([np.inf, -np.inf], np.nan).dropna().max(),
    1,
)



fig = px.scatter(
    df_with_stations,
    x='Capacity',
    y='Model Boardings',
    title='Observed vs Modeled',
    hover_data={'name': True, 'Model Boardings': True, 'Capacity': True, 'Boardings/Capacity': ':.2f'},
    color='color',
)

fig.add_shape(
    type='line',
    x0=0,
    y0=0,
    x1=max_val,
    y1=max_val,
    line=dict(color='black', width=1, dash='dash'),
)

fig.update_layout(
    height=800,
    width=800,
    xaxis=dict(range=[0, max_val], scaleanchor='y', scaleratio=1),
    yaxis=dict(range=[0, max_val], scaleanchor='x', scaleratio=1),
)
fig.show()

In [15]:
print(df_with_stations[['XCoord', 'YCoord']].describe())

             XCoord         YCoord
count  1.450000e+02     145.000000
mean   1.278003e+06  216892.358621
std    4.400226e+04   95881.518263
min    1.190151e+06   34324.529412
25%    1.241443e+06  146281.352941
50%    1.285435e+06  214675.647059
75%    1.306688e+06  280149.117647
max    1.404196e+06  455719.588235


In [16]:
gdf = gpd.GeoDataFrame(df_with_stations, geometry=gpd.points_from_xy(df_with_stations.XCoord, df_with_stations.YCoord), crs='EPSG:2285')
gdf.to_crs('EPSG:4326', inplace=True)

fig = px.scatter_mapbox(gdf, lat=gdf.geometry.y, lon=gdf.geometry.x, color='Boardings/Capacity', 
                        size='Boardings/Capacity', hover_data=['name','TAZ','Model Boardings', 'Capacity'], 
                        title='Park and Ride Usage vs Capacity', zoom=8)
fig.update_layout(mapbox_style="carto-positron")
fig.update_layout(margin={"r":0,"t":0,"l":0,"b":0})

fig.show()

NameError: name 'gpd' is not defined

In [ ]:
# # get observed data
p_r_observe_data = pd.read_sql(text("SELECT * FROM observed_park_and_ride"), con=conn.connect())


In [ ]:
p_r_observe_data.head()

,prasset_id,data_year,lot_name,lot_address,city,county,subarea,ownership_status,capacity,occupancy,utilization_rate,x_coord,y_coord,notes,ObjectId
0,1,2023,Shoreline P&R,18821 Aurora Ave N,Shoreline,King,North King County,Permanent,393,197,0,1268242,283392,None,1
1,328,2023,Edmonds Station Leased Lot Salish Crossings,212 Railroad Ave,Edmonds,Snohomish,Snohomish County,Leased,103,5,0,1259222,299663,Added in 2014,2
2,142,2022,Burien Transit Center,14900 4th Ave SW,Burien,King,South King County,Permanent,488,36,0,1268256,174822,Closed 10/12/10 for construction; Reopened 8/2...,3
3,4,2023,Shoreline United Methodist Church,14511 25th Ave NE,Shoreline,King,North King County,Leased,20,2,0,1278992,271384,None,4
4,5,2023,Kenmore P&R,7346 NE Bothell Way,Kenmore,King,North King County,Permanent,603,146,0,1294155,279616,None,5


In [ ]:
# filter to latest data record for each park and ride (only 2021 and later years)
idx = p_r_observe_data.groupby(['lot_name'])['data_year'].transform(max) == p_r_observe_data['data_year']
df_p_r_observe_data = p_r_observe_data[idx].copy()
df_p_r_observe_data = df_p_r_observe_data.loc[df_p_r_observe_data['data_year']>=2021].copy()

C:\Users\modeller\AppData\Local\Temp\ipykernel_33444\4243972432.py:2: FutureWarning:

The provided callable <built-in function max> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.



In [ ]:
# add station names to node file
df_p_r_nodes = pnr_capacity.merge(pd.DataFrame(stations), left_on="ZoneID", right_on="id")

In [ ]:
# transform to geodataframe
gdf_p_r_nodes = gpd.GeoDataFrame(
    df_p_r_nodes, geometry=gpd.points_from_xy(df_p_r_nodes.XCoord, df_p_r_nodes.YCoord), crs="EPSG:2285"
)

gdf_p_r_observe_data = gpd.GeoDataFrame(
    df_p_r_observe_data, geometry=gpd.points_from_xy(df_p_r_observe_data.x_coord, df_p_r_observe_data.y_coord), crs="EPSG:2285"
)

In [ ]:
# merge with park and ride name
gdf_p_r_data_name = gdf_p_r_nodes.merge(gdf_p_r_observe_data[['data_year', 'lot_name', 'lot_address', 'city','capacity', 'occupancy']],
                                        left_on="name",right_on="lot_name")

In [ ]:
# merge by nearest location
gdf_p_r_data_near = gpd.sjoin_nearest(gdf_p_r_nodes.loc[~(gdf_p_r_nodes["name"].isin(gdf_p_r_data_name.name))],
                                      gdf_p_r_observe_data[['data_year', 'lot_name', 'lot_address', 'city','capacity', 'occupancy', 'geometry']],
                                      max_distance=500)

df_col = ['NodeID', 'ZoneID', 'data_year', 'name', 
          'lot_name', 'lot_address', 'city', 'Capacity', 'capacity', 'occupancy', 
          'XCoord', 'YCoord', 'geometry']
gdf_p_r_data = pd.concat([gdf_p_r_data_name[df_col],gdf_p_r_data_near[df_col]]).copy()

In [ ]:
# merge boardings
gdf_p_r_data = gdf_p_r_data.merge(df, left_on="ZoneID", right_on="TAZ")

In [ ]:
gdf_p_r_data.columns

Index(['NodeID_x', 'ZoneID', 'data_year', 'name', 'lot_name', 'lot_address',
       'city', 'Capacity_x', 'capacity', 'occupancy', 'XCoord_x', 'YCoord_x',
       'geometry', 'person_id', 'tour_type', 'tour_type_count',
       'tour_type_num', 'tour_num', 'tour_count', 'tour_category',
       'number_of_participants', 'destination', 'origin', 'household_id',
       'tdd', 'start', 'end', 'duration', 'composition', 'destination_logsum',
       'pnr_zone_id', 'tour_mode', 'mode_choice_logsum',
       'tour_distance_one_way', 'atwork_subtour_frequency', 'parent_tour_id',
       'stop_frequency', 'primary_purpose', 'MAZ', 'TAZ_x', 'NodeID_y', 'taz',
       'XCoord_y', 'YCoord_y', 'Capacity_y', 'Cost', 'TAZ_y',
       'Model Boardings', 'Boardings/Capacity'],
      dtype='object')

In [ ]:
gdf_p_r_data = gdf_p_r_data[['NodeID_x', 'ZoneID', 'data_year', 'name', 'lot_name', 'lot_address',
       'city', 'Capacity_x', 'capacity', 'occupancy', 'XCoord_x', 'YCoord_x',
       'geometry', 'Model Boardings']].copy()

In [ ]:
gdf_p_r_data['Model Boardings - occupancy'] = gdf_p_r_data['Model Boardings'] - gdf_p_r_data['occupancy']
gdf_p_r_data['percent diff'] = gdf_p_r_data['Model Boardings - occupancy'] / gdf_p_r_data['occupancy']

gdf_p_r_data.to_crs('EPSG:4326', inplace=True)


In [ ]:
fig = px.scatter_mapbox(gdf_p_r_data, lat=gdf_p_r_data.geometry.y, lon=gdf_p_r_data.geometry.x, color='percent diff', 
                        hover_data=['name','ZoneID','Model Boardings', 'Capacity_x', 'Model Boardings - occupancy'], 
                        title='Park and Ride Usage vs Observed Data', zoom=8)
fig.update_layout(mapbox_style="carto-positron")
fig.update_layout(margin={"r":0,"t":0,"l":0,"b":0})

fig.show()

C:\Users\modeller\AppData\Local\Temp\ipykernel_33444\1241491485.py:1: DeprecationWarning:

*scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/

